# Evaluation Suite

This notebook evaluates retrieval quality with common IR metrics (`Precision@k`, `Recall@k`, `MRR@k`).
It is organized to run end-to-end in a stable and reproducible way.


## Scope

This notebook compares retrieval methods under the same protocol:
1. Retrieve top-k documents per query.
2. Convert retrieved indices to document IDs.
3. Compute average `Precision@k`, `Recall@k`, and `MRR@k`.
4. Compare quality and latency across models.


In [ ]:
import sys
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

cwd = Path.cwd()
project_root = cwd if (cwd / "src").exists() else cwd.parent
sys.path.insert(0, str(project_root))

from src.data.load import load_all
from src.data.preprocess import add_content_field
from src.evaluation.evaluate import evaluate_run, indices_to_docids, adapt_ground_truth


## Experiment Configuration

Project-level hyperparameters are defined once and reused in all sections.


In [ ]:
K_VALUES = [5, 10, 20, 50]
TEXT_FIELD = "content"
EMBEDDING_MODELS = [
    "all-MiniLM-L6-v2",
    "all-MiniLM-L12-v2",
    "multi-qa-mpnet-base-dot-v1",
]
BATCH_SIZE = 64
BM25_METHODS = ["plus", "okapi"]

RAW_DIR = project_root / "data" / "raw"
PROCESSED_DIR = project_root / "data" / "processed"
CACHE_DIR = project_root / "data" / "cache"
USE_EMBEDDING_CACHE = False  # If False, recompute embeddings and skip cache read/write.

# Keep runtime practical in notebook execution.
MAX_DOCS = 5000
RUN_KAGGLE_EXPORTS = False  # Set True to generate all Kaggle CSVs (heavy).

print("Embedding models:", EMBEDDING_MODELS)
print("BM25 methods:", BM25_METHODS)
print("USE_EMBEDDING_CACHE:", USE_EMBEDDING_CACHE)
print("RUN_KAGGLE_EXPORTS:", RUN_KAGGLE_EXPORTS)


## Data Setup

The dataset is prepared with a unified `content` field for documents and queries,
plus relevance labels (`gt`) and ordered `query_ids` for evaluation.


In [ ]:
# Data source
docs_path = PROCESSED_DIR / "docs_with_content.json"
queries_path = PROCESSED_DIR / "queries_train_with_content.json"

if docs_path.exists() and queries_path.exists():
    with open(docs_path, "r", encoding="utf-8") as f:
        docs = json.load(f)
    with open(queries_path, "r", encoding="utf-8") as f:
        queries_train = json.load(f)
else:
    docs_raw, queries_train_raw, _, _ = load_all(RAW_DIR)
    docs, queries_train = add_content_field(docs_raw, queries_train_raw, clean=True)

with open(RAW_DIR / "qgts_train.json", "r", encoding="utf-8") as f:
    qgts_train = json.load(f)

query_ids = [str(q["id"]) for q in queries_train]
gt_full = adapt_ground_truth(qgts_train)

# Build a deterministic doc subset that always keeps relevant documents.
doc_by_id = {str(d["id"]): d for d in docs}
relevant_ids = {
    doc_id
    for qid in query_ids
    for doc_id in gt_full.get(str(qid), [])
    if doc_id in doc_by_id
}

if MAX_DOCS is not None:
    selected_ids = set(relevant_ids)
    if len(selected_ids) < MAX_DOCS:
        for d in docs:
            did = str(d["id"])
            if did in selected_ids:
                continue
            selected_ids.add(did)
            if len(selected_ids) >= MAX_DOCS:
                break
else:
    selected_ids = {str(d["id"]) for d in docs}

docs = [d for d in docs if str(d["id"]) in selected_ids]

# Filter GT to the current candidate document set.
gt = {
    str(qid): [doc_id for doc_id in gt_full.get(str(qid), []) if doc_id in selected_ids]
    for qid in query_ids
}

non_empty_q = sum(1 for qid in query_ids if len(gt.get(str(qid), [])) > 0)

print(f"docs={len(docs)}, queries={len(queries_train)}, gt_queries={len(gt)}")
print(f"queries_with_relevant_docs_in_candidate_set={non_empty_q}")



## Embeddings Evaluation

This section runs embeddings retrieval for multiple SentenceTransformer models,
then computes evaluation metrics at multiple `k` values.


In [ ]:
# Embeddings block
from src.retrieval.embeddings import build_embeddings, retrieve_embeddings

rows = []

# Keep k values valid for current dataset size.
k_eval = [k for k in K_VALUES if k <= len(docs)]
if not k_eval:
    raise ValueError("No valid k for current docs size.")

embedding_debug_examples = {}

for model_name in EMBEDDING_MODELS:
    print(f"\n=== Embeddings model: {model_name} ===")

    t_fit = time.perf_counter()
    doc_emb, query_emb = build_embeddings(
        docs,
        queries_train,
        text_field=TEXT_FIELD,
        model_name=model_name,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        cache_dir=CACHE_DIR,
        use_cache=USE_EMBEDDING_CACHE,
    )
    fit_time = time.perf_counter() - t_fit

    print("doc_emb shape:", doc_emb.shape)
    print("query_emb shape:", query_emb.shape)

    last_topk_indices = None

    for k in k_eval:
        t1 = time.perf_counter()
        topk_indices, topk_scores = retrieve_embeddings(doc_emb, query_emb, k=k)
        retrieval_time = time.perf_counter() - t1

        pred_docids = indices_to_docids(topk_indices, docs)
        metrics = evaluate_run(pred_docids, gt, query_ids, k)

        rows.append({
            "model": "embeddings",
            "embedding_model": model_name,
            "bm25_method": None,
            "k": k,
            "fit_time_s": fit_time,
            "retrieval_time_s": retrieval_time,
            **metrics,
        })

        last_topk_indices = topk_indices

    if last_topk_indices is not None and len(last_topk_indices) > 0:
        sample_pred_docids = indices_to_docids(last_topk_indices, docs)
        embedding_debug_examples[model_name] = sample_pred_docids[0][:min(10, k_eval[0])]

if embedding_debug_examples:
    print("\nEmbeddings sample doc_ids for Q1 by model:")
    for model_name, sample_ids in embedding_debug_examples.items():
        print(f"- {model_name}: {sample_ids}")


## TF-IDF and BM25 Evaluation

This section computes TF-IDF and BM25 runs and evaluates them with the same metrics
for direct comparison with embeddings.


In [ ]:
# TF-IDF and BM25 blocks
from src.retrieval.tfidf import fit_tfidf, retrieve_tfidf
from src.retrieval.bm25 import fit_bm25, retrieve_bm25

# TF-IDF
t_tfidf_fit = time.perf_counter()
tfidf_vectorizer, tfidf_doc_matrix = fit_tfidf(docs, text_field=TEXT_FIELD)
tfidf_fit_time = time.perf_counter() - t_tfidf_fit

for k in k_eval:
    t1 = time.perf_counter()
    tfidf_topk_indices, tfidf_topk_scores = retrieve_tfidf(
        tfidf_vectorizer,
        tfidf_doc_matrix,
        queries_train,
        k=k,
        text_field=TEXT_FIELD,
    )
    tfidf_retrieval_time = time.perf_counter() - t1

    tfidf_pred_docids = indices_to_docids(tfidf_topk_indices, docs)
    tfidf_metrics = evaluate_run(tfidf_pred_docids, gt, query_ids, k)

    rows.append({
        "model": "tfidf",
        "embedding_model": None,
        "bm25_method": None,
        "k": k,
        "fit_time_s": tfidf_fit_time,
        "retrieval_time_s": tfidf_retrieval_time,
        **tfidf_metrics,
    })

# BM25 variants: BM25+ and BM25 Okapi
bm25_debug_examples = {}
for bm25_method in BM25_METHODS:
    print(f"\n=== BM25 method: {bm25_method} ===")

    t_bm25_fit = time.perf_counter()
    bm25_model = fit_bm25(docs, text_field=TEXT_FIELD, method=bm25_method)
    bm25_fit_time = time.perf_counter() - t_bm25_fit

    last_bm25_pred_docids = None

    for k in k_eval:
        t1 = time.perf_counter()
        bm25_topk_indices, bm25_topk_scores = retrieve_bm25(
            bm25_model,
            docs,
            queries_train,
            k=k,
            text_field=TEXT_FIELD,
        )
        bm25_retrieval_time = time.perf_counter() - t1

        bm25_pred_docids = indices_to_docids(bm25_topk_indices, docs)
        bm25_metrics = evaluate_run(bm25_pred_docids, gt, query_ids, k)

        rows.append({
            "model": "bm25",
            "embedding_model": None,
            "bm25_method": bm25_method,
            "k": k,
            "fit_time_s": bm25_fit_time,
            "retrieval_time_s": bm25_retrieval_time,
            **bm25_metrics,
        })

        last_bm25_pred_docids = bm25_pred_docids

    if last_bm25_pred_docids is not None:
        bm25_debug_examples[bm25_method] = last_bm25_pred_docids[0][:min(10, k_eval[0])]

print("TF-IDF sample doc_ids for Q1:", tfidf_pred_docids[0][:min(10, k_eval[0])])
if bm25_debug_examples:
    print("BM25 sample doc_ids for Q1 by method:")
    for m, sample_ids in bm25_debug_examples.items():
        print(f"- {m}: {sample_ids}")


## Results and Analysis

Use the tables below to analyze:
1. Which model performs best per metric (including each embedding model and each BM25 variant).
2. How increasing `k` changes precision/recall/MRR.
3. Quality versus latency trade-offs.
4. Final model choice based on your objective.


In [ ]:
# Results tables
raw_results_df = pd.DataFrame(rows)

if "embedding_model" not in raw_results_df.columns:
    raw_results_df["embedding_model"] = ""
if "bm25_method" not in raw_results_df.columns:
    raw_results_df["bm25_method"] = ""
if "fit_time_s" not in raw_results_df.columns:
    raw_results_df["fit_time_s"] = np.nan

raw_results_df["embedding_model"] = raw_results_df["embedding_model"].fillna("")
raw_results_df["bm25_method"] = raw_results_df["bm25_method"].fillna("")

# Build a readable model label.
def _model_label(row):
    if row["model"] == "embeddings":
        return f"embeddings:{row['embedding_model']}"
    if row["model"] == "bm25":
        return f"bm25:{row['bm25_method']}"
    return row["model"]

raw_results_df["model_label"] = raw_results_df.apply(_model_label, axis=1)

# Keep the comparison table compact: one `model` column and one `model_label` column.
results_df = raw_results_df[
    [
        "model",
        "model_label",
        "k",
        "precision@k",
        "recall@k",
        "mrr@k",
        "fit_time_s",
        "retrieval_time_s",
    ]
].sort_values(["model", "model_label", "k"]).reset_index(drop=True)

display(results_df)

quality_cols = ["precision@k", "recall@k", "mrr@k"]
print("=== Quality by model and k ===")
display(results_df.pivot(index="k", columns="model_label", values=quality_cols))

print("=== Retrieval time (s) by model and k ===")
display(results_df.pivot(index="k", columns="model_label", values="retrieval_time_s"))

# Optional helper: best mrr@k model for each k
best_by_k = (
    results_df.sort_values(["k", "mrr@k"], ascending=[True, False])
    .groupby("k", as_index=False)
    .first()[["k", "model", "model_label", "precision@k", "recall@k", "mrr@k", "retrieval_time_s"]]
)
print("=== Best model by MRR@k ===")
display(best_by_k)

# Export artifacts for reporting and reproducibility.
runs_dir = project_root / "outputs" / "runs"
runs_dir.mkdir(parents=True, exist_ok=True)

results_csv = runs_dir / "evaluation_suite_results.csv"
best_csv = runs_dir / "evaluation_suite_best_by_k.csv"

results_df.to_csv(results_csv, index=False)
best_by_k.to_csv(best_csv, index=False)

print(f"Saved: {results_csv}")
print(f"Saved: {best_csv}")


## Kaggle Submissions Export

This section generates one Kaggle-ready CSV per retrieval method used in this notebook
(TF-IDF, BM25 variants, and each embedding model) on `queries_test`.

This section is optional and is controlled by `RUN_KAGGLE_EXPORTS` (default: `False`).


In [ ]:
if not RUN_KAGGLE_EXPORTS:
    print("Kaggle submissions export skipped (RUN_KAGGLE_EXPORTS=False).")
else:
    from src.kaggle.format import save_submission
    from src.retrieval.tfidf import fit_tfidf, retrieve_tfidf
    from src.retrieval.bm25 import fit_bm25, retrieve_bm25
    from src.retrieval.embeddings import build_embeddings, retrieve_embeddings

    # Always export submissions on full corpus + test queries.
    docs_processed_path = PROCESSED_DIR / "docs_with_content.json"
    queries_test_processed_path = PROCESSED_DIR / "queries_test_with_content.json"

    if docs_processed_path.exists() and queries_test_processed_path.exists():
        with open(docs_processed_path, "r", encoding="utf-8") as f:
            docs_submit = json.load(f)
        with open(queries_test_processed_path, "r", encoding="utf-8") as f:
            queries_test = json.load(f)
    else:
        docs_raw, _, queries_test_raw, _ = load_all(RAW_DIR)
        docs_submit, queries_test = add_content_field(docs_raw, queries_test_raw, clean=True)

    submission_query_ids = [str(q["id"]) for q in queries_test]
    submission_top_k = min(100, len(docs_submit))

    submissions_dir = project_root / "outputs" / "submissions" / "by_method"
    submissions_dir.mkdir(parents=True, exist_ok=True)

    export_rows = []

    def _save_method_submission(pred_docids: list[list[str]], method_label: str) -> None:
        output_path = submissions_dir / f"submission_{method_label}_top{submission_top_k}.csv"
        save_submission(
            query_ids=submission_query_ids,
            pred_docids=pred_docids,
            output_path=output_path,
            top_k=submission_top_k,
            category="?",
        )
        export_rows.append({
            "model_label": method_label,
            "rows": len(submission_query_ids),
            "top_k": submission_top_k,
            "path": str(output_path),
        })

    # TF-IDF submission
    tfidf_vectorizer, tfidf_doc_matrix = fit_tfidf(docs_submit, text_field=TEXT_FIELD)
    tfidf_topk_indices, _ = retrieve_tfidf(
        tfidf_vectorizer,
        tfidf_doc_matrix,
        queries_test,
        k=submission_top_k,
        text_field=TEXT_FIELD,
    )
    tfidf_pred_docids = indices_to_docids(tfidf_topk_indices, docs_submit)
    _save_method_submission(tfidf_pred_docids, "tfidf")

    # BM25 submissions (plus + okapi)
    for bm25_method in BM25_METHODS:
        bm25_model = fit_bm25(docs_submit, text_field=TEXT_FIELD, method=bm25_method)
        bm25_topk_indices, _ = retrieve_bm25(
            bm25_model,
            docs_submit,
            queries_test,
            k=submission_top_k,
            text_field=TEXT_FIELD,
        )
        bm25_pred_docids = indices_to_docids(bm25_topk_indices, docs_submit)
        _save_method_submission(bm25_pred_docids, f"bm25_{bm25_method}")

    # Embeddings submissions (one per model)
    for model_name in EMBEDDING_MODELS:
        safe_model_name = model_name.replace("/", "_")

        doc_emb, query_emb = build_embeddings(
            docs_submit,
            queries_test,
            text_field=TEXT_FIELD,
            model_name=model_name,
            batch_size=BATCH_SIZE,
            show_progress_bar=True,
            cache_dir=CACHE_DIR,
            use_cache=USE_EMBEDDING_CACHE,
        )

        emb_topk_indices, _ = retrieve_embeddings(doc_emb, query_emb, k=submission_top_k)
        emb_pred_docids = indices_to_docids(emb_topk_indices, docs_submit)
        _save_method_submission(emb_pred_docids, f"embeddings_{safe_model_name}")

    exports_df = pd.DataFrame(export_rows).sort_values("model_label").reset_index(drop=True)
    manifest_path = project_root / "outputs" / "runs" / "submission_exports_from_notebook05.csv"
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    exports_df.to_csv(manifest_path, index=False)

    print(f"Saved {len(exports_df)} method submissions in: {submissions_dir}")
    print(f"Saved export manifest: {manifest_path}")

    display(exports_df)



## Final Conclusion

**Selected final submission setup (Phase 1):** `hybrid_bm25_embeddings` (weighted RRF fusion of **BM25+** and **embeddings**) in `notebooks/06_phase1_submission.ipynb`, with **`TOP_K = 100`** for Kaggle format.

**Why this final setup?**
- This notebook (`05`) compares the main retrieval families (**TF-IDF**, **BM25**, **embeddings**) under a shared evaluation protocol.
- The final submission notebook (`06`) uses a **hybrid fusion** strategy to combine lexical and semantic signals, which is a stronger practical choice for Kaggle submission than a single-model baseline.
- In practice, BM25 and embeddings are complementary: BM25 captures exact lexical overlap well, while embeddings improve semantic matching.

**How to read the results in this notebook (`05`)**
- Use the tables here to compare the base methods on `precision@k`, `recall@k`, `mrr@k`, and retrieval latency.
- Use these comparisons to justify why a hybrid fusion is reasonable for the final submission pipeline.

**Why not submit a single-model baseline directly?**
- A single model (TF-IDF, BM25, or embeddings) is simpler, but it relies on only one retrieval signal.
- The hybrid approach is designed to be more robust across query types by combining complementary rankings.

**Latency observations:**
- TF-IDF is generally fast.
- BM25 is slower in the current notebook setup.
- Embeddings retrieval can be fast at query time once embeddings are available/cached, but the first run may incur model loading/encoding cost.
- Hybrid retrieval adds fusion overhead, but can improve ranking quality in exchange.

**Limitations:**
- First execution can be slower due to model download/cache warmup (`data/cache`).
- Evaluation is done on train queries and a capped candidate set (`MAX_DOCS=5000`), so final Kaggle behavior may differ.
- Offline environments require the embedding model to be pre-cached locally.
